# Compare_notebook – GazeCapture: Training, Baselines und Hyperparameter-Vergleich

Dieses Notebook bündelt alle Basis-Experimente auf GazeCapture (ResNet-18, ImageNet-vortrainiert, 4-Schicht-MLP-Kopf).
Es nutzt ausschließlich die Projektmodule (`dataset.py`, `model.py`, `metrics.py`, `exp.py`, `spatial_analysis.py`),
damit Notebook, Skripte und Masterarbeit **dieselbe** Implementierung verwenden.

| Abschnitt | Inhalt | Abbildung/Tabelle im Kapitel *Experiments & Results* |
|---|---|---|
| 1 | Setup, Konfiguration | – |
| 2 | Rohdaten vs. normalisierte Ziele | Fig. 1a |
| 3 | Baselines (gesamter Datensatz **und** gleiche Teilmenge wie das Modell) | Tab. 1 |
| 4 | Einzelner Trainingslauf mit Lernkurve + korrekten Baselines | Fig. 1b |
| 5 | Hyperparameter-Sweeps (Batch, LR, Layer, Aktivierung, Loss, Größe, Split) mit 98 %-KI | Tab. 2–8 |
| 6 | Lernkurven verschiedener Lernraten | Fig. 2 |
| 7 | Räumliche Fehleranalyse (Zentrum/Rand/Ecke) und Loss-Vergleich | Fig. 4, Tab. 9 |
| 8 | Diagnose: Kollaps der Vorhersagen zum Bildschirmzentrum | Text Abschn. „Discussion“ |

### Metrik (verbindlich)
$$E_\text{diag} = 100\cdot\frac{\sqrt{\tfrac1N\sum_i \lVert \hat{\mathbf p}_i-\mathbf p_i\rVert_2^2}}{\sqrt2}\ [\%]$$
Euklidischer RMSE geteilt durch die Bildschirmdiagonale $\sqrt2$ im Einheitsquadrat.

> **Bugfix (Okt. 2026):** Frühere Versionen nutzten `sqrt(mean_squared_error(t, p))/√2`. sklearn mittelt für 2-D-Ziele über x **und** y,
> daher waren alle Modellfehler **um den Faktor √2 zu klein**, die Baselines dagegen korrekt. Alte Logwerte werden in Abschnitt 5 exakt mit √2 umgerechnet.

## 1. Setup

In [ ]:
import os, sys, json, re
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Notebook liegt in notebooks/ -> Projektwurzel als Arbeitsverzeichnis verwenden
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT))

from metrics import diagonal_error_pct, legacy_diagonal_error_pct, subset_baselines
from dataset import build_dataloaders, build_embedding_dataloaders, subset_targets, load_normalized_targets
from model import (build_model, build_head, build_frozen_backbone, train_model, evaluate, predict,
                   confidence_interval)
import exp
import spatial_analysis as sa

FIG_DIR = Path("results/figures"); FIG_DIR.mkdir(parents=True, exist_ok=True)
print("Projektwurzel:", ROOT)

In [ ]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CONFIG = dict(exp.DEFAULT_CONFIG)   # identisch zu exp.py
# Empfehlung für NEUE Läufe (keine Modellauswahl auf dem Testset, zufällige Teilmenge):
# CONFIG.update(sample_seed=0, val_size=200)
print(device); CONFIG

## 2. Rohdaten vs. normalisierte Blickziele (Fig. 1a)
Die Rohkoordinaten hängen vom Gerät ab (Bildschirmgröße, Ausrichtung). Durch $x_\text{norm}=x/W$, $y_\text{norm}=y/H$ liegen alle Ziele im Einheitsquadrat.

In [ ]:
raw = pd.read_csv("dataset/labels.csv")
norm = pd.read_csv("dataset/norm_labels.csv")
fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
ax[0].scatter(raw["x"], raw["y"], s=1, alpha=0.2); ax[0].set_title("Raw targets (device units)")
ax[1].scatter(norm["x"], norm["y"], s=1, alpha=0.2); ax[1].set_title("Normalized targets [0,1]²")
ax[1].set_xlim(0, 1); ax[1].set_ylim(0, 1)
for a in ax: a.set_xlabel("x"); a.set_ylabel("y")
plt.tight_layout(); plt.savefig(FIG_DIR / "fig1a_raw_vs_normalized.png", dpi=200); plt.show()
print(f"{len(norm):,} Samples, {norm['subject_ID'].nunique() if 'subject_ID' in norm else 'n/a'} Probanden")

## 3. Baselines
* **Konstant:** sagt für jedes Testbild den Mittelwert der *Trainings*ziele voraus.
* **Zufällig:** gleichverteilte Vorhersagen in $[0,1]^2$.

**Wichtig:** Eine Baseline ist nur dann eine faire Referenzlinie in einer Lernkurve, wenn sie auf **derselben Test-Teilmenge** berechnet wird wie das Modell.
Die Werte von `exp.compute_baselines` (Bootstrap über den *gesamten* Datensatz) sind ein globaler Referenzwert, aber nicht direkt mit einer 1000/200-Teilmenge vergleichbar.

In [ ]:
full = exp.compute_baselines(repeats=30)          # gesamter Datensatz (Bootstrap)
print("Gesamter Datensatz:", {k: tuple(round(x, 2) for x in v) for k, v in full.items()})

rows = []
for split in ("norm_subject", "norm_random"):
    for tr, te in ((1000, 200), (10000, 2000)):
        for seed in (None, 0):
            a, b = subset_targets(split, tr, te, sample_seed=seed)
            bl = subset_baselines(a, b)
            rows.append(dict(split=split, train=tr, test=te,
                             subset="erste N Zeilen" if seed is None else f"zufällig (seed {seed})",
                             constant=round(bl["constant"], 2), random=round(bl["random"], 2)))
pd.DataFrame(rows)

**Hinweis zu den Teilmengen:** Mit `sample_seed=None` werden die *ersten* N Zeilen der Split-CSV genommen. Da die CSVs nach Proband sortiert sind,
stammen z. B. die ersten 1000 Trainingsbilder nur von 5 Probanden und die ersten 200 Testbilder (norm_subject) von nur 1 Proband.
Für neue Experimente wird `sample_seed` empfohlen.

In [ ]:
for split in ("norm_subject", "norm_random"):
    tr_df = pd.read_csv(f"splits/{split}_train.csv"); te_df = pd.read_csv(f"splits/{split}_test.csv")
    print(split, "| Probanden train[:1000] =", tr_df.iloc[:1000].subject_ID.nunique(),
          "| test[:200] =", te_df.iloc[:200].subject_ID.nunique())

## 4. Einzelner Trainingslauf und Lernkurve (Fig. 1b)

**Warum startet die Kurve bei Epoche 0 unter der (globalen) Konstant-Baseline?**
1. Epoche 0 ist die Auswertung **vor** dem ersten Optimierungsschritt. Der frisch initialisierte Kopf liefert nahezu konstante Werte, die Sigmoid-Ausgabe liegt bei ≈ (0,5; 0,5). Das untrainierte Modell *ist* also praktisch ein konstanter „Bildschirmmitte“-Prädiktor.
2. Die früher eingezeichnete Linie (27,25 %) wurde auf dem **gesamten** Datensatz berechnet, die Kurve aber auf einer 200-Bilder-Teilmenge mit eigener Verteilung (z. B. 26,7 % auf `norm_random` 1000/200).
3. In älteren Notebook-Versionen kam zusätzlich der √2-Fehler der Metrik hinzu.

Korrektur: Die Baselines werden jetzt mit `subset_baselines` auf genau den Trainings-/Testdaten der Kurve berechnet.

In [ ]:
def single_run(cfg, epochs=15, patience=1000, lr=None, verbose=True):
    # Ein Lauf mit fester Epochenzahl (patience groß) -> vollständige Lernkurve
    lr = lr or cfg["learning_rate"]
    loaders = build_dataloaders(cfg["split_type"], cfg["train_size"], cfg["test_size"], cfg["batch_size"],
                                num_workers=cfg.get("num_workers", 0), sample_seed=cfg.get("sample_seed"),
                                val_size=cfg.get("val_size", 0))
    model, name = build_model(activation=cfg["activation"])
    res = train_model(model, name, loaders[0], loaders[1], device, epochs=epochs, learning_rate=lr,
                      weight_decay=cfg["weight_decay"], patience=patience,
                      trainable_layers=cfg["trainable_layers"], loss_name=cfg["loss"],
                      verbose=verbose, val_loader=loaders[2] if len(loaders) == 3 else None)
    return res, model, loaders

run_cfg = {**CONFIG, "split_type": "norm_random"}
res, model, loaders = single_run(run_cfg, epochs=15)

In [ ]:
a, b = subset_targets(run_cfg["split_type"], run_cfg["train_size"], run_cfg["test_size"], run_cfg.get("sample_seed"))
bl = subset_baselines(a, b)
ep = np.arange(len(res["test_errors"]))
plt.figure(figsize=(9, 5))
plt.plot(ep, res["train_errors"], lw=2, label="Train error")
plt.plot(ep, res["test_errors"], lw=2, label="Test error")
if res["val_errors"]: plt.plot(ep, res["val_errors"], lw=2, label="Validation error")
plt.axhline(bl["constant"], color="tab:green", ls="--", label=f"Constant baseline (same test subset): {bl['constant']:.2f}%")
plt.axhline(bl["random"], color="tab:red", ls="--", label=f"Random baseline (same test subset): {bl['random']:.2f}%")
plt.scatter([0], [res["test_errors"][0]], color="k", zorder=5)
plt.annotate("epoch 0 = untrained model\n(≈ constant centre predictor)", (0, res["test_errors"][0]),
             xytext=(1.5, res["test_errors"][0] + 3), arrowprops=dict(arrowstyle="->"))
plt.xlabel("Epoch"); plt.ylabel("Diagonal error (%)"); plt.legend(fontsize=8); plt.grid(alpha=0.3)
plt.title(f"{run_cfg['split_type']} {run_cfg['train_size']}/{run_cfg['test_size']}, lr={run_cfg['learning_rate']}")
plt.savefig(FIG_DIR / "fig1b_learning_curve_baselines.png", dpi=200); plt.show()

t, p = predict(model, loaders[1], device)
print("Epoche-0-Testfehler:", res["epoch0_test_error"], "| Std. der Vorhersagen (x, y):", p.std(0).round(4))

## 5. Hyperparameter-Sweeps (Tab. 2–8)
Jeder Sweep variiert **eine** Achse, alle anderen Werte bleiben auf `DEFAULT_CONFIG`; 5 Wiederholungen, 98 %-t-Konfidenzintervall.
Die Sweeps sind teuer → standardmäßig werden **gespeicherte Ergebnisse** geladen. Zum Neu-Rechnen `RUN_SWEEPS = True` setzen
(oder im Terminal: `python exp.py --experiment all --repeats 5`).

In [ ]:
RUN_SWEEPS = False
if RUN_SWEEPS:
    experiments = exp.build_experiments(repeats=5, base_config=CONFIG, device=device)
    sweep_results = {name: fn() for name, fn in experiments.items()}
    Path("results/exp_reproduction").mkdir(parents=True, exist_ok=True)
    json.dump(sweep_results, open("results/exp_reproduction/exp_results_new.json", "w"), indent=2, default=float)

### 5.1 Historisches Log `results_experiment_all` einlesen und umrechnen
Das Log wurde **vor** dem Metrik-Fix erzeugt. Modellwerte und KI-Grenzen werden exakt mit √2 multipliziert (lineare Skalierung, kein Informationsverlust).
Die Baselines im Log (27,28 % / 39,82 %) waren schon korrekt und werden **nicht** umgerechnet.

In [ ]:
LOG_FILE = Path("results/exp_reproduction/results_experiment_all.txt")   # Kopie des Logs hier ablegen
pat = re.compile(r"\[(\w+)\] (\S+): mean_test_error=([\d.]+)% CI=98%=\[([\d.]+), ([\d.]+)\] mean_time=([\d.]+)min")

def parse_log(path, convert=True):
    f = np.sqrt(2) if convert else 1.0
    rows = []
    for m in pat.finditer(Path(path).read_text(encoding="utf-8")):
        sweep, cfg, mean, lo, hi, t = m.groups()
        rows.append(dict(sweep=sweep, config=cfg, mean=float(mean) * f, ci_low=float(lo) * f,
                         ci_high=float(hi) * f, time_min=float(t)))
    return pd.DataFrame(rows)

if LOG_FILE.exists():
    log_df = parse_log(LOG_FILE)
    display(log_df.round(2))
else:
    print("Log nicht gefunden:", LOG_FILE)

In [ ]:
# Selected option of each sweep (DEFAULT_CONFIG). Marked with \\sel (dagger) in the tables.
SELECTED = {"batch_size": "32", "learning_rate": "0.0001", "trainable_layers": "fc_only",
            "activation": "Sigmoid", "loss_function": "standard", "dataset_size": "1000_200",
            "split_type": "norm_subject"}
from scipy import stats as st
T98 = st.t.ppf(0.99, 4)                      # 5 runs, 98 % two-sided CI

def welch_p(r1, r2, n=5):
    sd = lambda r: (r.ci_high - r["mean"]) / T98 * np.sqrt(n)   # SD aus dem 98%-KI
    return st.ttest_ind_from_stats(r1["mean"], sd(r1), n, r2["mean"], sd(r2), n, equal_var=False).pvalue

def latex_table(df, sweep, alpha=0.05):
    # Regel: Mittelwert der gewählten Option nur dann fett, wenn sie gegen JEDE
    # andere Option signifikant besser ist (Welch-t-Test p < alpha, kleinerer Mittelwert).
    d = df[df.sweep == sweep]
    sel = d[d.config == SELECTED[sweep]].iloc[0]
    others = d[d.config != SELECTED[sweep]]
    pvals = {i: welch_p(sel, r) for i, r in others.iterrows()}
    bold = all(p < alpha and sel["mean"] < r["mean"] for (i, p), (_, r) in zip(pvals.items(), others.iterrows()))
    for i, r in d.iterrows():
        is_sel = r.config == SELECTED[sweep]
        m = f"{r['mean']:.2f}"
        if is_sel and bold: m = f"\\textbf{{{m}}}"
        p = "---" if is_sel else f"{pvals[i]:.3f}"
        print(f"{r.config}{'\\sel{}' if is_sel else ''} & {m} & [{r.ci_low:.2f}, {r.ci_high:.2f}] & {p} & {r.time_min:.3f} \\\\")
    print(f"% bold = {bold}")

if LOG_FILE.exists():
    for s in log_df.sweep.unique():
        print("%", s); latex_table(log_df, s)

### 5.2 Vergleich mit der Konstant-Baseline derselben Teilmenge
Ohne diesen Vergleich ist eine Tabelle von Mittelwerten nicht interpretierbar.

In [ ]:
ref = {}
for split in ("norm_subject", "norm_random"):
    for tr, te in ((1000, 200), (10000, 2000)):
        a, b = subset_targets(split, tr, te); ref[(split, tr, te)] = subset_baselines(a, b)["constant"]
print({k: round(v, 2) for k, v in ref.items()})
if LOG_FILE.exists():
    d = log_df.copy()
    d["const_same_subset"] = ref[("norm_subject", 1000, 200)]
    d.loc[d.config == "10000_2000", "const_same_subset"] = ref[("norm_subject", 10000, 2000)]
    d.loc[d.config == "norm_random", "const_same_subset"] = ref[("norm_random", 1000, 200)]
    d["model_minus_const"] = d["mean"] - d["const_same_subset"]
    display(d.round(2))

## 6. Lernkurven verschiedener Lernraten (Fig. 2)
Feste 15 Epochen ohne Early Stopping, damit die Kurvenformen vergleichbar sind.

In [ ]:
RUN_LR_CURVES = False
if RUN_LR_CURVES:
    lr_curves = {}
    for lr in (1e-3, 1e-4, 1e-5, 1e-6):
        r, _, _ = single_run(run_cfg, epochs=15, lr=lr, verbose=False)
        lr_curves[lr] = r["test_errors"]
    plt.figure(figsize=(9, 6))
    for lr, c in lr_curves.items():
        plt.plot(range(len(c)), c, marker="o", ms=3, label=f"lr = {lr:g}")
    plt.axhline(bl["constant"], color="gray", ls="--", label="constant baseline (same subset)")
    plt.xlabel("Epoch (0 = untrained)"); plt.ylabel("Test diagonal error (%)"); plt.legend(); plt.grid(alpha=0.3)
    plt.savefig(FIG_DIR / "fig2_learning_rates.png", dpi=200); plt.show()

## 7. Räumliche Fehleranalyse (Fig. 4) und Loss-Vergleich (Tab. 9)

**Bildschirmaufteilung** im normierten Raum $[0,1]^2$ (3×3-Referenzgitter):
* Zentrum: $(0{,}5;0{,}5)$
* Ränder: $(0{,}1;0{,}5),(0{,}5;0{,}1),(0{,}9;0{,}5),(0{,}5;0{,}9)$
* Ecken: $(0{,}1;0{,}1),(0{,}1;0{,}9),(0{,}9;0{,}1),(0{,}9;0{,}9)$

Ein Testpunkt gehört zu einem Gitterpunkt $g$, wenn $\lVert\mathbf p-g\rVert_2\le r=0{,}15$.
Heatmap: $H(x,y)=\bar e(c)\,\exp\!\big(-\lVert(x,y)-c\rVert^2/(2\sigma^2)\big)$ mit $\sigma=0{,}20$; die Farbe zeigt den lokalen mittleren euklidischen Fehler (gleiche Skala in allen Teilbildern).

In [ ]:
gt, pred = sa.load_predictions("results/heatmap/base_loss1.csv")      # Standard L1
gt2, pred2 = sa.load_predictions("results/heatmap/base_loss2.csv")    # Sensitive L1
sa.plot_figure4(gt, pred, FIG_DIR / "fig4_spatial_heatmaps.png")
g1, g2 = sa.grid_statistics(gt, pred), sa.grid_statistics(gt2, pred2)
g1["sensitive"] = g2["mean_error"]; g1["delta"] = g1["sensitive"] - g1["mean_error"]
display(g1.round(4))
print("Regionen (L1):", sa.region_statistics(gt, pred))
print("Regionen (Sensitive):", sa.region_statistics(gt2, pred2))
from IPython.display import Image as IPImage; IPImage(str(FIG_DIR / "fig4_spatial_heatmaps.png"))

### Loss-Funktionen (Definitionen wie in `model.py`)
* Standard L1 (`nn.L1Loss`): $\mathcal L_{1}=\frac{1}{2N}\sum_i\big(|\hat x_i-x_i|+|\hat y_i-y_i|\big)$
* Smooth L1 (`nn.SmoothL1Loss`, β = 1): quadratisch für $|r|<\beta$, sonst linear
* Sensitive L1: $\mathcal L_{s}=\frac1N\sum_i w_i\,\tfrac12\big(|\hat x_i-x_i|+|\hat y_i-y_i|\big)$ mit $w_i=1+|x_i-0{,}5|+|y_i-0{,}5|\in[1,2]$

## 8. Diagnose: Kollaps der Vorhersagen

In [ ]:
for f in sorted(Path("results/heatmap").glob("*.csv")):
    g, p = sa.load_predictions(f)
    print(f"{f.name:28s} model={diagonal_error_pct(g, p):6.2f}%  centre-constant="
          f"{diagonal_error_pct(g, np.full_like(g, 0.5)):6.2f}%  std(pred)={p.std(0).round(4)}  std(gt)={g.std(0).round(3)}")

**Interpretation:** Die Standardabweichung der Vorhersagen ist ~100-mal kleiner als die der Zielwerte; der Modellfehler entspricht dem eines konstanten Mittelpunkt-Prädiktors.
In den `exp.py`-Sweeps (eingefrorener ResNet-18, Vollbilder, 1000 Trainingsbilder von 5 Probanden) lernt das GazeCapture-Modell damit praktisch nur den Mittelwert der Ziele.
(Einzelne Notebook-Läufe auf `norm_random` erreichten kurzzeitig ≈ 23,6 % gegenüber 26,7 % Konstant-Baseline – ein kleiner, aber realer Lerneffekt.)
Empfohlene nächste Schritte: zufällige Teilmengen (`sample_seed`), Validierungsset (`val_size`), Augen-/Gesichtsausschnitte statt Vollbild, mehr Probanden, Feature-Normalisierung.